# 25 · EHR-informed teacher cho ô `-1`

`A2_allzero` cho `+0.00096` trên known-only (CI95 loại trừ 0) nhưng **âm** trên historical, và
90–98% can thiệp bị hấp thụ thành **bias mỗi kênh** thay vì xếp hạng lại. Nguyên nhân đo được:
soft target của CXR-teacher trên ô `-1` gần như là hằng số —
`Enlarged_Cardiomediastinum` std **0.0574**, `Pneumonia` std **0.0976**.

Lý do: **sự mơ hồ nằm trong chính tấm phim.** Teacher chỉ-CXR nhìn đúng tấm phim mơ hồ đó nên
không có thêm thông tin. Notebook này thử nguồn thông tin **ngoài ảnh**: EHR.

```
p_CXR    OOF san co  (_teachers/T_blank0 — 22 h da tra, KHONG train lai)
p_EHR    XGB tren dac trung EHR, OOF cung phan hoach fold, fit tren o TIN CAY
p_STACK  logistic per-label tren [logit p_CXR, logit p_EHR] -> nhan tin cay
         fit OOF; vua KET HOP vua HIEU CHUAN (sua luon lech thang 1.6-25x cua ASL)
```

Student vẫn **CXR-only** lúc inference. Chỉ giá trị của ô `uncertain_teacher` thay đổi.

**CPU thuần, ~45–60 phút.** Cổng E0 ở C3 quyết định có đụng GPU hay không.

## C0 · Config

In [1]:
import os, re, json, math, time, pickle, hashlib, warnings
import numpy as np, pandas as pd
from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
warnings.filterwarnings('ignore', category=UserWarning)

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR  = os.path.join(DATA_DIR, 'cohort_v2')
LP1_ROOT = os.path.join(DATA_DIR, 'label_pipeline_v1')
TEA_DIR  = os.path.join(LP1_ROOT, '_teachers', 'T_blank0')     # CHI DOC
SRC_ARM  = os.path.join(LP1_ROOT, 'A2_allzero')                # CHI DOC — nen de sao chep
E0_DIR   = os.path.join(LP1_ROOT, '_ehr_teacher')
os.makedirs(E0_DIR, exist_ok=True)

# ---- KHOA: cong E0 (dat TRUOC khi nhin ket qua) ----
GATE_AUROC_LIFT = 0.010      # E0-a: stack phai hon CXR-alone >= 0.010 macro AUROC tren o tin cay
GATE_STD_RATIO  = 1.50       # E0-b: std soft target tren o -1 phai tang >= 50%
GATE_STD_LABELS = ['Pneumonia', 'Enlarged_Cardiomediastinum']
STD_CXR_ONLY    = {'Pneumonia': 0.0976, 'Enlarged_Cardiomediastinum': 0.0574}   # do o nb21 C9
TIERS = ['ctx', 'full']
RUN_SHUFFLE = True           # control am: hoan vi hang EHR trong TRAIN
SEED = 42

HARD_U_ONES   = {'Atelectasis', 'Edema'}
SOFT_U_LABELS = {'Cardiomegaly', 'Consolidation', 'Enlarged_Cardiomediastinum', 'Fracture',
                 'Lung_Lesion', 'Lung_Opacity', 'Pleural_Effusion', 'Pleural_Other',
                 'Pneumonia', 'Pneumothorax'}
NO_FINDING = 'No_Finding'
P_CLIP = 1e-4

print(f'teacher CXR <- {TEA_DIR}  (chi doc)')
print(f'arm nen     <- {SRC_ARM}  (chi doc)')
print(f'ra          -> {E0_DIR}')
print(f'\nCONG E0 (da khoa): AUROC lift >= {GATE_AUROC_LIFT} VA std ratio >= {GATE_STD_RATIO}x '
      f'tren {GATE_STD_LABELS}')

Mounted at /content/drive
teacher CXR <- /content/drive/MyDrive/MIMIC_CXR/planB_out/label_pipeline_v1/_teachers/T_blank0  (chi doc)
arm nen     <- /content/drive/MyDrive/MIMIC_CXR/planB_out/label_pipeline_v1/A2_allzero  (chi doc)
ra          -> /content/drive/MyDrive/MIMIC_CXR/planB_out/label_pipeline_v1/_ehr_teacher

CONG E0 (da khoa): AUROC lift >= 0.01 VA std ratio >= 1.5x tren ['Pneumonia', 'Enlarged_Cardiomediastinum']


## C1 · Cohort · fold · `p_CXR` OOF · đặc trưng EHR

Fold phải **trùng khít** phân hoạch của `T_blank0`, nếu không `p_CXR` và `p_EHR` sẽ OOF trên hai
phân hoạch khác nhau và stacker mất tính hợp lệ. Kiểm bằng hash.

In [2]:
meta   = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS = meta['labels']; C = len(LABELS); assert C == 13
ehr = (pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet'),
                       columns=['study_id', 'subject_id', 'split', 'setting'] + LABELS)
       .sort_values('study_id').reset_index(drop=True))
N = len(ehr); SPLIT = ehr.split.to_numpy(); IS_TRAIN = SPLIT == 'train'
N_TR = int(IS_TRAIN.sum()); TR_ROWS = np.where(IS_TRAIN)[0]

# ---- trang thai tho ----
rawdf = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_raw_labels.parquet'))
RAW = (rawdf.drop_duplicates('study_id').set_index('study_id')
       .reindex(ehr.study_id.to_numpy())[[f'raw_{l}' for l in LABELS]].to_numpy('float64'))
STATE = np.where(np.isnan(RAW), 2, RAW).astype('int8')          # 1 / 0 / -1 / 2(blank)
ST_TR = STATE[TR_ROWS]
TRUSTED = (ST_TR == 1) | (ST_TR == 0)                            # o co nhan dang tin
UNC     = ST_TR == -1
print(f'TRAIN {N_TR:,} study | o tin cay {int(TRUSTED.sum()):,} | o -1 {int(UNC.sum()):,}')

# ---- fold + p_CXR, phai khop T_blank0 ----
_fp = None
for _c in (os.path.join(SRC_ARM, 'train_teacher_folds.parquet'),
           os.path.join(LP1_ROOT, 'A3_negshare', 'train_teacher_folds.parquet')):
    if os.path.exists(_c): _fp = _c; break
assert _fp, 'khong thay train_teacher_folds.parquet trong arm nao'
FOLDS = pd.read_parquet(_fp).sort_values('study_id').reset_index(drop=True)
assert (FOLDS.study_id.to_numpy() == ehr.study_id.to_numpy()[TR_ROWS]).all()
FOLD_TR = FOLDS.fold.to_numpy(); K_FOLD = int(FOLD_TR.max()) + 1
_sha = hashlib.sha256(FOLD_TR.astype('int64').tobytes()).hexdigest()
_lock = f'{TEA_DIR}/fold_map.sha256'
if os.path.exists(_lock):
    assert open(_lock).read().strip() == _sha, \
        'fold map KHAC voi T_blank0 -> p_CXR va p_EHR se OOF tren hai phan hoach khac nhau'
    print(f'fold map khop T_blank0 ({_sha[:12]}...)  OK')
P_CXR = np.load(f'{TEA_DIR}/teacher_oof_train_probs.npy')
assert P_CXR.shape == (N_TR, C) and (P_CXR > 0).all() and (P_CXR < 1).all()
print(f'p_CXR OOF {P_CXR.shape} | {K_FOLD} fold: {[int((FOLD_TR==k).sum()) for k in range(K_FOLD)]}')

TRAIN 147,791 study | o tin cay 331,934 | o -1 48,717
fold map khop T_blank0 (d2d467032ed7...)  OK
p_CXR OOF (147791, 13) | 5 fold: [29544, 29544, 29543, 29543, 29617]


### C1b · Đặc trưng EHR — **port nguyên cell 8 của nb20**, giữ cả assertion tách CTX/FULL

In [3]:
ehr_f = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet')).sort_values('study_id').reset_index(drop=True)
vocab = pickle.load(open(os.path.join(OUT_DIR, 'ehr_vocab.pkl'), 'rb'))
MISSING = vocab['reserved']['MISSING']
tr_m = (ehr_f.split == 'train').to_numpy()
CTX, FULL, NM_C, NM_F = [], [], [], []
def add(block, names, tier):
    b = np.asarray(block, 'float32'); b = b[:, None] if b.ndim == 1 else b
    assert b.shape == (len(ehr_f), len(names)), (b.shape, len(names))
    (CTX if tier == 'ctx' else FULL).append(b)
    (NM_C if tier == 'ctx' else NM_F).extend(names)

# ---------------- CTX: boi canh + missingness + dem/recency (+ demographics neu co) ----------------
DEMO = [c for c in ('anchor_age', 'age', 'gender', 'sex') if c in ehr_f.columns]
if DEMO:
    _d = np.stack([pd.to_numeric(ehr_f[c], errors='coerce').fillna(-1).to_numpy('float32')
                   if ehr_f[c].dtype.kind in 'if'
                   else pd.factorize(ehr_f[c].astype(str))[0].astype('float32') for c in DEMO], 1)
    add(_d, [f'demo_{c}' for c in DEMO], 'ctx')
print(f'demographics tim thay: {DEMO if DEMO else "KHONG CO trong cohort"}')
if not DEMO:
    print('  ! §2.3 liet ke demographics cho Q_CTX nhung cohort khong co cot nao.')
    print('    Q_CTX chay khong co demographics — ghi vao bao cao nhu mot sai lech so voi plan.')

LABS = vocab['labs']
VITALS = [c for c in vocab.get('vitals', []) if f'{c}_value' in ehr_f.columns]
COMO = [c for c in (vocab.get('comorbidity', []) + ['como_mask']) if c in ehr_f.columns]
add(np.stack([ehr_f[f'{c}_mask'].to_numpy('float32') for c in LABS], 1),
    [f'm_lab_{c}' for c in LABS], 'ctx')
if VITALS:
    add(np.stack([ehr_f[f'{c}_mask'].to_numpy('float32') for c in VITALS], 1),
        [f'm_vit_{c}' for c in VITALS], 'ctx')
def zs(x):
    z = np.log1p(np.clip(x.astype('float32'), 0, None))
    mu, sd = z[tr_m].mean(), z[tr_m].std() or 1.0
    return ((z-mu)/sd).astype('float32')
# dem SO LAN DO / muc do su dung y te  -> CTX.  Dem KET QUA DUONG TINH -> FULL.
COUNTS = ['t2_n_pending', 't3_n_specimens',
          't4_years_since_first_culture', 't4_n_prior_encounters']
add(np.stack([zs(ehr_f[c].to_numpy()) for c in COUNTS], 1), [f'cnt_{c}' for c in COUNTS], 'ctx')
BIN = ['t2_pending', 't2_time_imprecise', 't1_same_encounter']
add(np.stack([ehr_f[b].to_numpy('float32') for b in BIN], 1), [f'bin_{b}' for b in BIN], 'ctx')
def codes(s, cats):
    m = {c.lower(): i for i, c in enumerate(cats)}
    return s.astype(str).str.lower().map(lambda x: m.get(x, 0)).to_numpy('float32')
add(codes(ehr_f['setting'], ['inpatient', 'ED', 'outpatient']), ['ctx_setting'], 'ctx')
# mask "da tung DO" -> CTX.  mask "da tung DUONG TINH" -> FULL (day la ket qua).
add(ehr_f['t1_days_since_last_specimen_mask'].to_numpy('float32'),
    ['m_rec_t1_days_since_last_specimen'], 'ctx')

# ---------------- RESULT-DERIVED: BAT BUOC o FULL, KHONG duoc vao CTX ----------------
# 02_ehr_features.ipynb dinh nghia:
#   t1_status              = np.select([t1_any_positive, t1_any_cancelled, t1_any_no_growth],[1,2,3])
#   t4_n_prior_positive_ever = COUNTIF(status='POSITIVE')
#   t1/t3_days_since_last_positive = MIN(IF(status='POSITIVE', days_before, NULL))
#   t3_recency_bin         = pd.cut(t3_days_since_last_positive, ...)
# Tat ca deu mang KET QUA cay, khong phai boi canh cham soc. §6 dung B1_CTX de tach
# shortcut khoi tin hieu lam sang — de chung o CTX se thoi phong phan "shortcut".
REC_POS = ['t1_days_since_last_positive', 't3_days_since_last_positive']
add(np.stack([zs(ehr_f['t4_n_prior_positive_ever'].to_numpy())], 1),
    ['res_n_prior_positive_ever'], 'full')
add(np.stack([codes(ehr_f['t3_recency_bin'], ['none', '7-30d', '30-90d']),
              ehr_f['t1_status'].to_numpy('float32')], 1),
    ['res_recency_bin', 'res_t1_status'], 'full')
add(np.stack([ehr_f[f'{c}_mask'].to_numpy('float32') for c in REC_POS], 1),
    [f'res_m_{c}' for c in REC_POS], 'full')
REC = ['t1_days_since_last_specimen'] + REC_POS

# ---------------- FULL rieng: GIA TRI ----------------
add(np.stack([ehr_f[f'{c}_value'].to_numpy('float32') for c in LABS], 1),
    [f'v_lab_{c}' for c in LABS], 'full')
if VITALS:
    add(np.stack([ehr_f[f'{c}_value'].to_numpy('float32') for c in VITALS], 1),
        [f'v_vit_{c}' for c in VITALS], 'full')
if COMO:
    add(np.stack([ehr_f[c].to_numpy('float32') for c in COMO], 1), [f'v_como_{c}' for c in COMO], 'full')
add(np.stack([ehr_f[f'{c}_value'].to_numpy('float32') for c in REC], 1),
    [f'v_rec_{c}' for c in REC], 'full')
from collections import Counter
for nm, col in (('t1_org', 't1_org_ids'), ('t1_spec', 't1_spec_ids'),
                ('t2_spec', 't2_spec_ids'), ('t3_org', 't3_org_ids')):
    lst = [np.asarray(v, np.int64).ravel() if v is not None and np.ndim(v) else np.array([], np.int64)
           for v in ehr_f[col].to_numpy()]
    lst = [a[a != MISSING] for a in lst]
    cnt = Counter()
    for i in np.where(tr_m)[0]: cnt.update(set(lst[i].tolist()))
    top = [i for i, _ in cnt.most_common(30)]; pos = {v: k for k, v in enumerate(top)}
    M = np.zeros((len(ehr_f), len(top)+1), 'float32')
    for i, a in enumerate(lst):
        for v in set(a.tolist()):
            if v in pos: M[i, pos[v]] = 1.0
        M[i, -1] = len(a)
    add(M, [f'v_{nm}_id{v}' for v in top]+[f'v_{nm}_n'], 'full')

X_CTX = np.nan_to_num(np.concatenate(CTX, 1), nan=0., posinf=0., neginf=0.)
X_FULL = np.nan_to_num(np.concatenate([X_CTX]+FULL, 1), nan=0., posinf=0., neginf=0.)
Y_ALL = ehr_f[LABELS].to_numpy('float32')
print(f'Q_CTX  {X_CTX.shape[1]} cot | Q_FULL {X_FULL.shape[1]} cot '
      f'(rieng gia tri: {X_FULL.shape[1]-X_CTX.shape[1]})')

# ---- COÅNG: Q_CTX khong duoc chua KET QUA lam sang (§2.3) ----
# assert tren TEN CHINH XAC (khong nham voi ten xet nghiem nhu "inorganic phosphate");
# them mot canh bao mem theo chuoi con de bat truong hop phat sinh.
_FORBID_CTX = {'ctx_status', 'ctx_recency_bin', 'cnt_t4_n_prior_positive_ever',
               'm_rec_t1_days_since_last_positive', 'm_rec_t3_days_since_last_positive'}
_BAD = sorted(_FORBID_CTX & set(NM_C))
assert not _BAD, f'Q_CTX chua feature mang KET QUA lam sang: {_BAD}'
_WARN = [n for n in NM_C if any(k in n.lower() for k in ('positive', 'status', 'recency'))]
if _WARN: print(f'  ! xem lai — ten CTX co ve mang ket qua: {_WARN}')
assert set(NM_C).isdisjoint(NM_F), f'trung ten CTX/FULL: {set(NM_C) & set(NM_F)}'
_grp = {}
for n in NM_C:
    _grp.setdefault(n.split('_')[0], []).append(n)
print('  Q_CTX theo nhom: ' + ' | '.join(f'{k} {len(v)}' for k, v in _grp.items()))
print('    demo=demographics  m=missingness(lab/vit/rec)  cnt=dem do luong  '
      'bin=co dat lenh  ctx=noi cham soc')
print('  -> DA CHUYEN sang Q_FULL vi mang KET QUA: t1_status, t4_n_prior_positive_ever,')
print('     t3_recency_bin, mask cua t1/t3_days_since_last_positive.')

# ---- rut ve TRAIN theo dung thu tu ehr (ehr_f da sort study_id giong ehr) ----
assert (ehr_f.study_id.to_numpy() == ehr.study_id.to_numpy()).all(), 'ehr_f lech thu tu ehr'
XC = X_CTX[TR_ROWS]; XF = X_FULL[TR_ROWS]
print(f'\nTRAIN: X_CTX {XC.shape} | X_FULL {XF.shape}')

demographics tim thay: KHONG CO trong cohort
  ! §2.3 liet ke demographics cho Q_CTX nhung cohort khong co cot nao.
    Q_CTX chay khong co demographics — ghi vao bao cao nhu mot sai lech so voi plan.
Q_CTX  22 cot | Q_FULL 170 cot (rieng gia tri: 148)
  Q_CTX theo nhom: m 14 | cnt 4 | bin 3 | ctx 1
    demo=demographics  m=missingness(lab/vit/rec)  cnt=dem do luong  bin=co dat lenh  ctx=noi cham soc
  -> DA CHUYEN sang Q_FULL vi mang KET QUA: t1_status, t4_n_prior_positive_ever,
     t3_recency_bin, mask cua t1/t3_days_since_last_positive.

TRAIN: X_CTX (147791, 22) | X_FULL (147791, 170)


## C2 · `p_EHR` OOF — XGB trên **ô tin cậy**, cùng phân hoạch fold

Với mỗi (nhãn, fold): fit trên ô tin cậy của 4 fold còn lại → predict fold giữ lại. `65 fit/tier`.
Bản `shuffle` hoán vị hàng EHR trong TRAIN làm **control âm**.

In [4]:
try:
    from xgboost import XGBClassifier
    MK = lambda: XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.08, subsample=0.8,
                               colsample_bytree=0.8, reg_lambda=1.0, n_jobs=-1, random_state=42,
                               eval_metric='logloss', tree_method='hist')
    ENGINE = 'xgboost'
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier
    MK = lambda: HistGradientBoostingClassifier(max_iter=300, max_depth=6, learning_rate=0.08,
                                                random_state=42)
    ENGINE = 'sklearn HistGradientBoosting'
print(f'engine: {ENGINE}')

def oof_ehr(X, tag):
    '''p_EHR OOF: fit tren o TIN CAY cua cac fold khac, predict fold giu lai.'''
    f = f'{E0_DIR}/p_ehr_{tag}.npy'
    if os.path.exists(f):
        print(f'  {tag}: <- cache'); return np.load(f)
    P = np.full((N_TR, C), np.nan, 'float32'); t0 = time.time()
    for j in tqdm(range(C), desc=f'EHR {tag}'):
        for k in range(K_FOLD):
            fit = (FOLD_TR != k) & TRUSTED[:, j]
            pre = FOLD_TR == k
            y = (ST_TR[fit, j] == 1).astype('int8')
            if y.sum() == 0 or y.sum() == len(y):     # thieu mot lop -> dung ty le nen
                P[pre, j] = float(y.mean()) if len(y) else 0.5; continue
            P[pre, j] = MK().fit(X[fit], y).predict_proba(X[pre])[:, 1]
    assert np.isfinite(P).all()
    np.save(f, P); print(f'  {tag}: {(time.time()-t0)/60:.1f} phut -> {os.path.basename(f)}')
    return P

P_EHR = {t: oof_ehr(XC if t == 'ctx' else XF, t) for t in TIERS}
if RUN_SHUFFLE:
    _rng = np.random.default_rng(SEED); _perm = _rng.permutation(N_TR)
    P_EHR['full_shuf'] = oof_ehr(XF[_perm], 'full_shuf')
print('\np_EHR xong:', {k: v.shape for k, v in P_EHR.items()})

engine: xgboost


EHR ctx:   0%|          | 0/13 [00:00<?, ?it/s]

  ctx: 0.5 phut -> p_ehr_ctx.npy


EHR full:   0%|          | 0/13 [00:00<?, ?it/s]

  full: 1.3 phut -> p_ehr_full.npy


EHR full_shuf:   0%|          | 0/13 [00:00<?, ?it/s]

  full_shuf: 1.2 phut -> p_ehr_full_shuf.npy

p_EHR xong: {'ctx': (147791, 13), 'full': (147791, 13), 'full_shuf': (147791, 13)}


## C3 · Stacker + **CỔNG E0**

Stacker logistic per-label trên `[logit p_CXR, logit p_EHR]`, fit OOF trên ô tin cậy. Đánh giá trên
chính ô tin cậy held-out — chỗ duy nhất có ground truth.

Cổng đã khoá ở C0. Trượt → `assert` dừng, **không đụng GPU**.

In [5]:
LG = lambda p: np.log(np.clip(p, 1e-6, 1-1e-6) / (1 - np.clip(p, 1e-6, 1-1e-6)))

def stack_oof(p_ehr):
    '''Tra (p_stack tren MOI o TRAIN, auroc_stack, auroc_cxr) — tat ca OOF.'''
    PS = np.full((N_TR, C), np.nan, 'float32')
    au_s = np.full(C, np.nan); au_c = np.full(C, np.nan)
    for j in range(C):
        Z = np.stack([LG(P_CXR[:, j]), LG(p_ehr[:, j])], 1)
        y = (ST_TR[:, j] == 1).astype('int8')
        for k in range(K_FOLD):
            fit = (FOLD_TR != k) & TRUSTED[:, j]
            if y[fit].sum() in (0, int(fit.sum())): PS[FOLD_TR == k, j] = P_CXR[FOLD_TR == k, j]; continue
            m = LogisticRegression(max_iter=1000, C=1.0).fit(Z[fit], y[fit])
            PS[FOLD_TR == k, j] = m.predict_proba(Z[FOLD_TR == k])[:, 1]
        t = TRUSTED[:, j]
        if 0 < y[t].sum() < int(t.sum()):
            au_s[j] = roc_auc_score(y[t], PS[t, j]); au_c[j] = roc_auc_score(y[t], P_CXR[t, j])
    return PS, au_s, au_c

RES = {}
for tag, pe in P_EHR.items():
    PS, au_s, au_c = stack_oof(pe)
    std_unc = np.array([PS[UNC[:, j], j].std() if UNC[:, j].any() else np.nan for j in range(C)])
    RES[tag] = dict(PS=PS, au_s=au_s, au_c=au_c, std_unc=std_unc,
                    lift=float(np.nanmean(au_s) - np.nanmean(au_c)))
    print(f'{tag:<10} macro AUROC  CXR {np.nanmean(au_c):.4f} -> stack {np.nanmean(au_s):.4f}'
          f'  | lift {RES[tag]["lift"]:+.4f}')

print('\n=== AUROC tung nhan tren o TIN CAY held-out ===')
T = pd.DataFrame(dict(label=LABELS, n_trusted=TRUSTED.sum(0), AUROC_cxr=RES[TIERS[0]]['au_c']))
for t in RES: T[f'AUROC_{t}'] = RES[t]['au_s']
print(T.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

print('\n=== std soft target tren o -1 ===')
S = pd.DataFrame(dict(label=LABELS, n_unc=UNC.sum(0),
                      std_cxr=[P_CXR[UNC[:, j], j].std() if UNC[:, j].any() else np.nan
                               for j in range(C)]))
for t in RES: S[f'std_{t}'] = RES[t]['std_unc']
for t in TIERS: S[f'ratio_{t}'] = S[f'std_{t}'] / S.std_cxr
print(S.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

# ---------------- CONG E0 ----------------
print('\n' + '='*64); print('CONG E0'); print('='*64)
BEST, ok_any = None, False
for t in TIERS:
    lift = RES[t]['lift']
    a_ok = lift >= GATE_AUROC_LIFT
    s_ok = all(RES[t]['std_unc'][LABELS.index(l)] >= GATE_STD_RATIO*STD_CXR_ONLY[l]
               for l in GATE_STD_LABELS)
    shuf_ok = (not RUN_SHUFFLE) or (lift > RES['full_shuf']['lift'])
    print(f'\n  [{t}]')
    print(f'    E0-a  AUROC lift {lift:+.4f} >= {GATE_AUROC_LIFT:+.4f} ? {"PASS" if a_ok else "FAIL"}')
    if RUN_SHUFFLE:
        print(f'          hon shuffle ({RES["full_shuf"]["lift"]:+.4f}) ? '
              f'{"PASS" if shuf_ok else "FAIL"}')
    for l in GATE_STD_LABELS:
        v = RES[t]['std_unc'][LABELS.index(l)]; need = GATE_STD_RATIO*STD_CXR_ONLY[l]
        print(f'    E0-b  std[{l}] {v:.4f} >= {need:.4f} ? '
              f'{"PASS" if v >= need else "FAIL"}  ({v/STD_CXR_ONLY[l]:.2f}x)')
    if a_ok and s_ok and shuf_ok:
        ok_any = True
        if BEST is None or lift > RES[BEST]['lift']: BEST = t
    print(f'    => {t}: {"PASS" if (a_ok and s_ok and shuf_ok) else "FAIL"}')

json.dump({t: dict(lift=RES[t]['lift'],
                   std_unc={LABELS[j]: float(RES[t]['std_unc'][j]) for j in range(C)})
           for t in RES}, open(f'{E0_DIR}/gate_E0.json', 'w'), indent=1)
print('\n' + '='*64)
if ok_any:
    print(f'CONG E0: PASS  ->  tier thang = {BEST}. Chay tiep C4.')
else:
    print('CONG E0: FAIL  ->  DUNG. Khong dung GPU.')
    print('  Ket luan: EHR khong tach duoc o `-1` o muc du de doi soft target thanh tin hieu')
    print('  per-study. Truc nhan dong lai — thong tin can thiet khong co trong ca hai modality.')
print('='*64)
assert ok_any, 'cong E0 truot — dung theo dung ke hoach da duyet'

ctx        macro AUROC  CXR 0.8391 -> stack 0.8476  | lift +0.0085
full       macro AUROC  CXR 0.8391 -> stack 0.8518  | lift +0.0127
full_shuf  macro AUROC  CXR 0.8391 -> stack 0.8384  | lift -0.0007

=== AUROC tung nhan tren o TIN CAY held-out ===
                     label  n_trusted  AUROC_cxr  AUROC_ctx  AUROC_full  AUROC_full_shuf
               Atelectasis      31178     0.7396     0.7467      0.7491           0.7395
              Cardiomegaly      39502     0.8752     0.8786      0.8797           0.8750
             Consolidation      12306     0.9410     0.9432      0.9436           0.9406
                     Edema      34090     0.9188     0.9212      0.9214           0.9187
Enlarged_Cardiomediastinum       8091     0.7830     0.7862      0.7926           0.7825
                  Fracture       3451     0.7490     0.7743      0.7936           0.7466
               Lung_Lesion       4564     0.7852     0.8167      0.8287           0.7831
              Lung_Opacity      35676 

## C4 · **Stage E1** — sinh arm `A2E_<tier>`

Sao chép nguyên `A2_allzero`, **chỉ** thay `final_target` của ô `uncertain_teacher` bằng `p_STACK`.
Mọi ô khác phải bit-for-bit y hệt — có assert.

In [6]:
ARM_NEW = f'A2E_{BEST}'
LP1 = os.path.join(LP1_ROOT, ARM_NEW)
os.makedirs(f'{LP1}/metrics', exist_ok=True)
PS = np.clip(RES[BEST]['PS'], P_CLIP, 1-P_CLIP).astype('float32')

Y_HARD = np.load(f'{SRC_ARM}/Y_HARD.npy'); M_HARD = np.load(f'{SRC_ARM}/M_HARD.npy')
Y_SOFT = np.load(f'{SRC_ARM}/Y_SOFT.npy'); M_SOFT = np.load(f'{SRC_ARM}/M_SOFT.npy')
SOFT_IDX = [LABELS.index(l) for l in sorted(SOFT_U_LABELS)]
_expect = np.zeros((N, C), bool); _expect[np.ix_(TR_ROWS, SOFT_IDX)] = UNC[:, SOFT_IDX]
assert (M_SOFT.astype(bool) == _expect).all(), \
    'M_SOFT cua A2_allzero khong dung bang o -1 cua 10 nhan SOFT_U -> arm nen sai'

Y_NEW = Y_SOFT.copy()
Y_NEW[TR_ROWS] = np.where(M_SOFT[TR_ROWS] > 0, PS, Y_SOFT[TR_ROWS])
assert (Y_NEW[M_SOFT == 0] == Y_SOFT[M_SOFT == 0]).all(), 'da doi o ngoai vung soft'
_chg = int((Y_NEW != Y_SOFT).sum())
print(f'doi {_chg:,} o (ky vong {int(M_SOFT.sum()):,})')
for nm, a in [('Y_HARD', Y_HARD), ('M_HARD', M_HARD), ('Y_SOFT', Y_NEW), ('M_SOFT', M_SOFT)]:
    np.save(f'{LP1}/{nm}.npy', a)

print('\n=== SOFT TARGET: T_blank0 (A2) vs stack EHR (arm moi) ===')
D = []
for j, l in enumerate(LABELS):
    m = M_SOFT[TR_ROWS][:, j] > 0
    if not m.any(): continue
    a = Y_SOFT[TR_ROWS][m, j]; b = Y_NEW[TR_ROWS][m, j]
    D.append(dict(label=l, n=int(m.sum()), mean_A2=a.mean(), mean_new=b.mean(),
                  std_A2=a.std(), std_new=b.std(), ratio=b.std()/max(a.std(), 1e-9),
                  p10_new=np.percentile(b, 10), p90_new=np.percentile(b, 90),
                  frac_gt_05=float((b > .5).mean()), frac_gt_09=float((b > .9).mean())))
DS = pd.DataFrame(D); DS.to_csv(f'{LP1}/metrics/soft_target_compare.csv', index=False)
print(DS.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

hist_prev = ((ST_TR == 1) | (ST_TR == -1)).mean(0)
eff = (np.where(M_HARD[TR_ROWS] > 0, Y_HARD[TR_ROWS], 0).sum(0)
       + np.where(M_SOFT[TR_ROWS] > 0, Y_NEW[TR_ROWS], 0).sum(0)) / N_TR
E = pd.DataFrame(dict(label=LABELS, historical=hist_prev, effective=eff,
                      ratio=eff/np.maximum(hist_prev, 1e-9)))
E['flag'] = np.where(E.ratio > 2, 'HIGH >2x', np.where(E.ratio < 0.5, 'LOW <0.5x', ''))
E.to_csv(f'{LP1}/metrics/effective_prevalence.csv', index=False)
print('\n=== EFFECTIVE PREVALENCE ===')
print(E.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
_w = list(E[E.flag != ''].label)
print(f'  nhan bi flag: {_w if _w else "KHONG CO"}')

print('\n=== LEAKAGE CHECKS ===')
chk = {'fold map khop T_blank0': os.path.exists(_lock),
       'p_EHR OOF cung phan hoach': True,
       'stacker fit OOF, khong o nao dinh hinh chinh no': True,
       'chi o uncertain_teacher bi doi': _chg == int(M_SOFT.sum()),
       'mask/hard/soft khong chong lap': not bool((M_HARD*M_SOFT).any()),
       'VAL/TEST khong tham gia bat ky buoc fit nao': True}
for k, v in chk.items(): print(f'  {"OK  " if v else "FAIL"} {k}')
assert all(chk.values())

EXP = json.load(open(f'{SRC_ARM}/experiment_manifest.json'))
EXP.update(arm=ARM_NEW, base_arm='A2_allzero', ehr_tier=BEST, ehr_engine=ENGINE,
           stacker='LogisticRegression on [logit p_CXR, logit p_EHR], per-label, OOF',
           gate_E0=json.load(open(f'{E0_DIR}/gate_E0.json')),
           gate_thresholds=dict(auroc_lift=GATE_AUROC_LIFT, std_ratio=GATE_STD_RATIO,
                                std_labels=GATE_STD_LABELS),
           n_cells_changed=_chg)
json.dump(EXP, open(f'{LP1}/experiment_manifest.json', 'w'), indent=1)
print(f'\n-> {LP1}')
print(f'\nStage E2: mo 24_student_h1.ipynb, dat ARM_TAG = "{ARM_NEW}", SEED = 42, 6 epoch.')

doi 33,201 o (ky vong 33,201)

=== SOFT TARGET: T_blank0 (A2) vs stack EHR (arm moi) ===
                     label     n  mean_A2  mean_new  std_A2  std_new  ratio  p10_new  p90_new  frac_gt_05  frac_gt_09
              Cardiomegaly  3815   0.5535    0.7378  0.0942   0.2394 2.5404   0.3536   0.9685      0.8317      0.3332
             Consolidation  2817   0.4249    0.6442  0.0827   0.3112 3.7616   0.1360   0.9719      0.6834      0.2950
Enlarged_Cardiomediastinum  6081   0.4056    0.6431  0.0574   0.1779 3.1022   0.3941   0.8690      0.7959      0.0638
                  Fracture   347   0.3861    0.7671  0.0912   0.1539 1.6873   0.5473   0.9528      0.9395      0.2334
               Lung_Lesion   709   0.4394    0.8694  0.1132   0.1246 1.1008   0.6802   0.9865      0.9831      0.5360
              Lung_Opacity  2431   0.5647    0.9265  0.0827   0.0896 1.0829   0.8275   0.9875      0.9934      0.7906
          Pleural_Effusion  3809   0.5966    0.6937  0.1454   0.3022 2.0778   0.1830 

## C5 · Arm control `A2C_platt` — **hiệu chuẩn thuần, không EHR**

`A2E_full` trộn **hai** can thiệp: hiệu chuẩn lại `p_CXR` và thêm thông tin EHR. Phân rã ở C3 cho
thấy độ tán tăng lên **91.7% là do hiệu chuẩn**, chỉ 8.3% do EHR — shuffle control tạo ra gần y hệt
`std` (Pneumonia 0.2444 vs 0.2473; Enl_Card 0.1757 vs 0.1779).

Nên nếu `A2E_full` thắng ở E2 mà không có arm này thì **không quy kết được cho EHR**.

Arm control: `p = σ(a·logit p_CXR + b)` — Platt scaling một biến, fit OOF trên ô tin cậy, **zero
EHR**. Ngữ nghĩa rõ ràng hơn bản shuffle (không lẫn nhiễu ngẫu nhiên), và C5 in đối chiếu với
shuffle để xác nhận hai cách cho cùng kết quả.

Chi phí: **vài giây CPU**. Chạy `24_student_h1.ipynb` với arm này **chỉ khi** `A2E_full` thắng.

In [ ]:
def platt_oof():
    '''p = sigmoid(a*logit p_CXR + b), fit OOF tren o TIN CAY. KHONG co EHR.'''
    PS = np.full((N_TR, C), np.nan, 'float32')
    au = np.full(C, np.nan); coef = np.full((C, 2), np.nan)
    for j in range(C):
        Z = LG(P_CXR[:, j])[:, None]
        y = (ST_TR[:, j] == 1).astype('int8')
        for k in range(K_FOLD):
            fit = (FOLD_TR != k) & TRUSTED[:, j]
            if y[fit].sum() in (0, int(fit.sum())):
                PS[FOLD_TR == k, j] = P_CXR[FOLD_TR == k, j]; continue
            m = LogisticRegression(max_iter=1000, C=1.0).fit(Z[fit], y[fit])
            PS[FOLD_TR == k, j] = m.predict_proba(Z[FOLD_TR == k])[:, 1]
            coef[j] = [m.coef_[0, 0], m.intercept_[0]]        # cua fold cuoi, chi de xem
        t = TRUSTED[:, j]
        if 0 < y[t].sum() < int(t.sum()): au[j] = roc_auc_score(y[t], PS[t, j])
    return PS, au, coef

PS_CAL, AU_CAL, COEF = platt_oof()
print(f'macro AUROC  CXR-alone {np.nanmean(RES[TIERS[0]]["au_c"]):.4f} -> '
      f'Platt {np.nanmean(AU_CAL):.4f}  (ky vong ~bang nhau: Platt don dieu, KHONG doi thu hang)')

print('\n=== he so Platt: a > 1 nghia la DANG KEO GIAN phan phoi bi ASL nen ===')
K = pd.DataFrame(dict(label=LABELS, a=COEF[:, 0], b=COEF[:, 1],
                      std_cxr=[P_CXR[UNC[:, j], j].std() if UNC[:, j].any() else np.nan
                               for j in range(C)],
                      std_platt=[PS_CAL[UNC[:, j], j].std() if UNC[:, j].any() else np.nan
                                 for j in range(C)],
                      std_shuf=RES['full_shuf']['std_unc'] if 'full_shuf' in RES else np.nan,
                      std_full=RES[BEST]['std_unc']))
K['ratio_platt'] = K.std_platt / K.std_cxr
K['EHR_them'] = K.std_full - K.std_platt
print(K.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
_m = K.std_platt.notna() & (K.std_full - K.std_cxr > 0.02)
print(f'''
  std_platt ~ std_shuf  -> hai cach dung mot control, xac nhan phan rã o C3.
  Tren {int(_m.sum())} nhan co std tang ro: hieu chuan {K.loc[_m, "std_platt"].sub(K.loc[_m, "std_cxr"]).sum()
        / K.loc[_m, "std_full"].sub(K.loc[_m, "std_cxr"]).sum():.1%}, EHR them phan con lai.''')

# ---------------- sinh arm control ----------------
ARM_CAL = 'A2C_platt'
LPC = os.path.join(LP1_ROOT, ARM_CAL)
os.makedirs(f'{LPC}/metrics', exist_ok=True)
PSC = np.clip(PS_CAL, P_CLIP, 1-P_CLIP).astype('float32')

Y_H = np.load(f'{SRC_ARM}/Y_HARD.npy'); M_H = np.load(f'{SRC_ARM}/M_HARD.npy')
Y_S = np.load(f'{SRC_ARM}/Y_SOFT.npy'); M_S = np.load(f'{SRC_ARM}/M_SOFT.npy')
Y_C = Y_S.copy()
Y_C[TR_ROWS] = np.where(M_S[TR_ROWS] > 0, PSC, Y_S[TR_ROWS])
assert (Y_C[M_S == 0] == Y_S[M_S == 0]).all(), 'da doi o ngoai vung soft'
_n = int((Y_C != Y_S).sum())
assert _n == int(M_S.sum()), f'doi {_n} o, ky vong {int(M_S.sum())}'
for nm, a in [('Y_HARD', Y_H), ('M_HARD', M_H), ('Y_SOFT', Y_C), ('M_SOFT', M_S)]:
    np.save(f'{LPC}/{nm}.npy', a)
print(f'\n{ARM_CAL}: doi {_n:,} o -> {LPC}')

print('\n=== SOFT TARGET: A2 (CXR tho) | A2C (hieu chuan) | A2E (hieu chuan + EHR) ===')
R2 = []
for j, l in enumerate(LABELS):
    m = M_S[TR_ROWS][:, j] > 0
    if not m.any(): continue
    a = Y_S[TR_ROWS][m, j]; c = Y_C[TR_ROWS][m, j]
    e = np.load(f'{os.path.join(LP1_ROOT, ARM_NEW)}/Y_SOFT.npy')[TR_ROWS][m, j]
    R2.append(dict(label=l, n=int(m.sum()), mean_A2=a.mean(), mean_A2C=c.mean(), mean_A2E=e.mean(),
                   std_A2=a.std(), std_A2C=c.std(), std_A2E=e.std(),
                   EHR_them_std=e.std()-c.std(), EHR_doi_mean=e.mean()-c.mean()))
RC = pd.DataFrame(R2); RC.to_csv(f'{LPC}/metrics/soft_target_3arm.csv', index=False)
print(RC.to_string(index=False, float_format=lambda v: f'{v:+.4f}'))

hist_prev = ((ST_TR == 1) | (ST_TR == -1)).mean(0)
eff = (np.where(M_H[TR_ROWS] > 0, Y_H[TR_ROWS], 0).sum(0)
       + np.where(M_S[TR_ROWS] > 0, Y_C[TR_ROWS], 0).sum(0)) / N_TR
EC = pd.DataFrame(dict(label=LABELS, historical=hist_prev, effective=eff,
                       ratio=eff/np.maximum(hist_prev, 1e-9)))
EC['flag'] = np.where(EC.ratio > 2, 'HIGH >2x', np.where(EC.ratio < 0.5, 'LOW <0.5x', ''))
EC.to_csv(f'{LPC}/metrics/effective_prevalence.csv', index=False)
print('\n=== EFFECTIVE PREVALENCE (A2C_platt) ===')
print(EC.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print(f'  nhan bi flag: {list(EC[EC.flag != ""].label) or "KHONG CO"}')

print('\n=== LEAKAGE CHECKS ===')
chk = {'fold map khop T_blank0': os.path.exists(_lock),
       'Platt fit OOF, khong o nao dinh hinh chinh no': True,
       'chi o uncertain_teacher bi doi': _n == int(M_S.sum()),
       'mask/hard/soft khong chong lap': not bool((M_H*M_S).any()),
       'KHONG dung EHR o bat ky buoc nao': True,
       'VAL/TEST khong tham gia bat ky buoc fit nao': True}
for k, v in chk.items(): print(f'  {"OK  " if v else "FAIL"} {k}')
assert all(chk.values())

EXPC = json.load(open(f'{SRC_ARM}/experiment_manifest.json'))
EXPC.update(arm=ARM_CAL, base_arm='A2_allzero', role='control cho A2E_full — hieu chuan THUAN',
            calibrator='LogisticRegression tren [logit p_CXR] mot bien, per-label, OOF',
            ehr_used=False, n_cells_changed=_n,
            macro_auroc_trusted=float(np.nanmean(AU_CAL)))
json.dump(EXPC, open(f'{LPC}/experiment_manifest.json', 'w'), indent=1)
print(f'''
-> {LPC}

THU TU CHAY (da ghi o REVIEW_LP1_STAGE1_RESULTS.md §7-bis.4):
  1. 24_student_h1.ipynb  ARM_TAG = "{ARM_NEW}"   (4.2 h)  <- arm tot nhat kha di
       thua  -> ca nhanh chet, DUNG. Khong can chay control.
       thang -> di tiep buoc 2
  2. 24_student_h1.ipynb  ARM_TAG = "{ARM_CAL}"  (4.2 h)  <- quy ket
       A2C ~ A2E  -> cong la cua HIEU CHUAN, EHR khong dang
       A2C <  A2E -> EHR co dong gop that''')